# Event loop

Event loop (или цикл событий) - основной необходимый компонент любой асинхронной программы. Цикл событий управляет конкурентным запуском корутин, запуская их по очереди. В момент, когда одна работающая корутина приостанавливает свое выполнение (с помощью конструкции await), например в ожидании получения данных по сети, event loop запускает следующую в очереди готовую к работе корутину (например, которая уже получила свой ответ и готова продолжать).

То есть, иными словами, event loop - это среда выполнения корутин в рамках одного потока в операционной системе.

Event loop выполняет несколько функций в асинхронной программе:

*   управляет запуском корутин
*   запускает callback-функции (в event loop можно запускать callback-функции в определённый момент времени. Например, по завершении работы конкретной корутины)
*   обрабатывает сетевой ввод/вывод
*   может запускать подпроцессы или потоки для выполнения в них блокирующих операций (Event loop работает в одном потоке ОС, но может управлять подпроцессами и потоками для блокирующих операций.)

Важно понимать, что когда мы запускаем асинхронную программу, мы по сути запускаем event loop, в котором “регистрируем” и запускаем основную корутину, которая в свою очередь, если это нужно, “регистрирует” другие корутины в этом event loop. Например, когда мы пишем asyncio.run(main()) - функция asyncio.run “под капотом”  сначала создает event loop, потом регистрирует в нем корутину main, и в конце запускает этот созданный event loop

## Работа с event loop

asyncio.new_event_loop() - создает и возвращает новый event loop

In [21]:
import asyncio

loop = asyncio.new_event_loop()

asyncio.get_running_loop() - с помощью этой функции можно получить объект event loop, если он уже запущен для текущего потока ОС. Если такого event loop нет, то функция выбросит исключение RuntimeError. 

Важный момент, данная функция может быть вызвана только из корутины или из callback-функции. Иначе: # RuntimeError: no running event loop

In [22]:
import asyncio

async def coro():
  loop = asyncio.get_running_loop()

asyncio.get_event_loop() - функция признана устаревшей (deprecated) в версии Python 3.10, но всё равно в коде часто можно её встретить. Возвращает существующий объект event loop. Если существующего нет, то автоматически создаст новый event loop и вернет его объект.

In [23]:
import asyncio

loop = asyncio.get_event_loop()

Несколько тонкостей при работе с event loop:

1.  Так как в разных ОС работа с неблокирующим вводом/выводом реализуется разными низкоуровневыми механизмами (например в Linux - epoll, в MacOS/BSD - kqueue, в Windows - IOCP), то в asyncio реализация класса event loop для разных ОС также отличается.
По умолчанию для операционных систем семейства Unix (включая MacOS) цикл событий -  это экземпляр класса asyncio.SelectorEventLoop, для Windows - asyncio.ProactorEventLoop.
Но главное то, что какого бы вида не был event loop, он обязательно реализует интерфейс класса asyncio.AbstractEventLoop, то есть предоставляет разработчикам единый (одинаковый для всех ОС) API для взаимодействия с циклом событий.

2.  Методы работы с объектом event loop напрямую, включая функции его получения рассмотренные выше, относятся к низкоуровневому API модуля asyncio. Поэтому в разработке asyncio-приложений стоит, если не избегать, то хотя бы минимизировать их использование.

## Запуск event loop

В asyncio приложениях мы запускаем именно event loop, а в нём уже запускаются корутины, которые мы “зарегистрировали” на запуск в цикле событий. Поэтому в большинстве случаев в контексте asyncio программ фразы “запустить event loop” и “запустить программу” будут иметь один и тот же смысл.

1.  Запуск через через функцию asyncio.run(<coroutine_object>). Это стандартный способ запуска event loop.

asyncio.run(...) делает следующее:
*   Всегда вначале создает цикл событий.
*   Принимает на вход объект корутины, и выполняет ее до завершения в созданном event loop.
*   После завершения работы, закрывает цикл событий.

Функцию asyncio.run() стоит использовать, как точку входа в программу, и в идеале запускать её единожды. В этом случае всегда есть основная корутина (обычно именуемая “main”), которая запускается в самом начале, а потом из нее уже плодятся новые корутины для запуска в event loop.

In [20]:
import asyncio

async def main():
  print("Started")
  await asyncio.sleep(3)
  print("Stopped")

if __name__ == '__main__':
  await main()
#   asyncio.run(main())

Started
Stopped


Далее приведем еще два способа запуска: loop.run_until_complete(...) и loop.run_forever().  Но они уже относятся к низкоуровневому API, поэтому не рекомендуется их использовать в asyncio-приложениях. 

Это методы объекта event loop, то есть для их использования нужен уже готовый объект event loop. 

Также, в отличие от варианта запуска с asyncio.run(), в этих случаях Вам самим нужно позаботиться вначале о создании, а после о корректном закрытии event loop по окончании работы программы.

2.  Метод loop.run_until_complete(<coroutine_object>)

Метод делает следующее:

*   Принимает на вход объект корутины.
*   Запускает цикл событий до тех пор, пока переданная корутина не завершится.
*   После этого цикл останавливается, но не закрывается автоматически.

In [ ]:
import asyncio

async def coro():
    await asyncio.sleep(1)
    return "Done"

# Вручную создаем цикл
loop = asyncio.new_event_loop()

try:
   
    # Запускаем цикл до завершения корутины
    result = loop.run_until_complete(coro())
    print(f"result: {result}")
   
finally:
    # Вручную закрываем цикл и освобождаем ресурсы.
    # Это ОЧЕНЬ важно делать, особенно на Windows!
    loop.close()

# После close() использовать этот loop нельзя.

3.  Метод loop.run_forever()

Что произойдет после вызова loop.run_forever():

*   Цикл событий запустится на неопределенное время. 
*   Цикл событий будет работать бесконечно, обрабатывая задачи и callback-функции, пока его явно не остановят.

! Остановить цикл можно с помощью метода loop.stop().

Данный метод запуска подходит, например, для создания серверов, которые должны постоянно работать.

В следующем примере используется пока не упомянутый в курсе метод loop.create_task(...), и может смутить не знакомых с ним читателей. Но не стоит из-за этого переживать, уже совсем скоро мы с ним познакомимся. Если кратко, с помощью loop.create_task(coroutine_object) мы можем “зарегистрировать” корутину coroutine_object для выполнения в цикле событий loop. 

In [ ]:
import asyncio

async def coro(loop):
  print("Start sleeping")
  await asyncio.sleep(1)
  print("Stop sleeping, stopping loop...")
  loop.stop() # Останавливаем event loop


# Создаем цикл событий
loop = asyncio.new_event_loop()

# "регистрируем" корутину на запуск в цикле событий
task = loop.create_task(coro(loop))


try:
  # Запускаем цикл НАВСЕГДА (пока его не остановят)
  loop.run_forever()
finally:
  loop.close()
  print("Event loop closed")